# The five-step method, run end to end

This notebook runs the five-step method of the WAISE paper on the repository's
encoding of the standards and shows each step's output inline:

> Patel and Jung, "Integrating Cybersecurity Into the AI Safety Assurance
> Argument: A GSN Pattern for AI-Based Perception Components in Highly Automated
> Driving", WAISE 2026 (workshop at SAFECOMP 2026), LNCS.

The companion position paper (Patel and Jung, "Operational Safety and
Cybersecurity Assurance of AI-Based Perception in Highly Automated Driving",
SAFECOMP 2026) covers operation and is not run here.

Sections whose heading says **"Repository analysis, not in either paper"** go
beyond the papers: severity scoring, the evidence-combination sketch, the
counterfactual analysis, the seed and threshold sensitivity checks and the
related-work comparison. They are this repository's own and are not results
either paper reports.

The notebook is a walkthrough, not the reproduction entry point; see
`REPRODUCING.md` for `make reproduce`. It runs in Google Colab or any Jupyter
environment and does not touch `data/empirical_results/`.

**The five steps (WAISE paper, Section 3.2):**
- **Step 1, claim extraction:** clauses of the four standards (ISO 26262,
  ISO 21448, ISO/SAE 21434, ISO/PAS 8800) and one informative technical report
  (ISO/IEC TR 5469) are read and turned into claims (52 clauses and 40 claims in
  this repository's encoding).
- **Step 2, lifecycle-phase mapping:** the claims are mapped onto six lifecycle
  phases.
- **Step 3, GSN construction:** the ISO/PAS 8800 Annex B pattern is extended to
  nine goals.
- **Step 4, junction-point analysis:** the output is the decision-point
  catalogue (WAISE Table 4).
- **Step 5, integration-induced finding identification:** the output is the
  findings classification (WAISE Table 5).

A seeded illustration of the weather-grid evaluation follows the five steps.
`data/synthetic_illustrations/README.md` states what its numbers are and are
not.

## 0. Environment setup

In [ ]:
# Install the repository (run this cell once)
import os
import sys

# Clone the repository if running in Colab
if 'google.colab' in sys.modules:
    if not os.path.exists('Assurance-Gaps-in-an-Integrated-Safety-and-Cybersecurity-Case'):
        !git clone https://github.com/milinpatel07/Assurance-Gaps-in-an-Integrated-Safety-and-Cybersecurity-Case.git
    os.chdir('Assurance-Gaps-in-an-Integrated-Safety-and-Cybersecurity-Case')
    !pip install -e ".[dev]" -q
    !apt-get install -y graphviz -qq
else:
    # Running locally -- ensure we're in the repo root
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    !pip install -e ".[dev]" -q

print(f"Working directory: {os.getcwd()}")
print("Setup complete.")

In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from IPython.display import display, HTML, Markdown

# Project imports
from src.standards.registry import StandardsRegistry
from src.gsn.integrated_pattern import build_integrated_gsn
from src.analysis.decision_points import DecisionPointCatalogue
from src.analysis.findings import GapClassification
from src.analysis.evidence_convergence import EvidenceConvergenceAnalysis
from src.evaluation.carla_evaluator import generate_synthetic_illustration
from src.evaluation.weather_conditions import generate_weather_grid, compute_triggering_coverage

# Paper identifiers for code identifiers: Gap-n is F-n, I-n is DP-n
import re
import textwrap

def paper_id(text):
    """Rewrite code ids (Gap-n, I-n) as the WAISE paper's ids (F-n, DP-n)."""
    text = re.sub(r'\bGap-(\d)', r'F-\1', text)
    return re.sub(r'\bI-(\d)\b', r'DP-\1', text)

def wrap(text, indent=4, width=88):
    """Print-ready text, wrapped at word boundaries rather than cut."""
    pad = ' ' * indent
    return textwrap.fill(' '.join(str(text).split()), width=width,
                         initial_indent=pad, subsequent_indent=pad)

# Set deterministic seed
SEED = 42
SCENES_PER_WEATHER = 50
np.random.seed(SEED)

print(f"Random seed: {SEED}")
print(f"Scenes per weather condition: {SCENES_PER_WEATHER}")

---
## Step 1: Claim extraction

The case study draws on four standards and one informative technical report:

| Document | Year | Scope |
|----------|------|-------|
| ISO 26262 | 2018 | Functional safety |
| ISO 21448 | 2022 | Safety of the intended functionality (SOTIF) |
| ISO/SAE 21434 | 2021 | Cybersecurity engineering |
| ISO/PAS 8800 | 2024 | Safety and artificial intelligence |
| ISO/IEC TR 5469 | 2024 | Functional safety and AI systems (informative technical report; an ISO/IEC JTC 1 document, not automotive-specific) |

Each document is read at clause level. A claim is a normative requirement, a
defined work product, or a required lifecycle activity; each extracted claim is
traceable to a specific clause (WAISE paper, Step 1). In this repository each
claim is also mapped to a GSN goal. The code counts all five documents, so the
printout below reports five.

In [ ]:
# Step 1: Build the standards registry and extract claims
registry = StandardsRegistry()

print("Documents read in Step 1 (four standards and one informative technical report)")
print("=" * 70)
total_claims = 0
total_clauses = 0

for std in registry.all_standards:
    n_claims = len(std.claims)
    n_clauses = len(std.clauses)
    total_claims += n_claims
    total_clauses += n_clauses
    print(f"\n  {std.standard_id} ({std.year}): {std.full_name}")
    print(f"    Clauses analysed: {n_clauses}")
    print(f"    Claims extracted: {n_claims}")
    for c in std.claims[:3]:
        goal = c.gsn_goal or '(outside the AI-component scope)'
        print(f"      [{c.claim_id}] -> {goal}")
        print(wrap(c.text, indent=10))
    if n_claims > 3:
        print(f"      ... and {n_claims - 3} more claims")

print(f"\n{'=' * 70}")
print(f"Total: {total_clauses} clauses -> {total_claims} claims from {len(registry.all_standards)} documents")

---
## Step 2: Lifecycle-phase mapping

The claims are mapped onto the six lifecycle phases of the WAISE paper (Step 2).
The grid spans the lifecycles of the four standards; it is not the phase model
of any one standard.

1. **Concept and requirements**: HARA, TARA, requirement specification
2. **Design and training**: architecture design, model training
3. **V&V**: verification and validation activities
4. **Integration and deployment**: system integration, release
5. **Operation and monitoring**: runtime monitoring, incident response
6. **Modification and re-assurance**: over-the-air (OTA) updates, re-assurance

The coverage matrix shows which documents address which phases.

In [ ]:
# Step 2: Compute the coverage matrix
coverage_matrix = registry.compute_coverage_matrix()

print("Coverage matrix: number of applicable clauses per lifecycle phase")
print("=" * 90)

phases = list(list(coverage_matrix.values())[0].keys())
documents = list(coverage_matrix.keys())
header = f"{'Lifecycle phase':<30}" + "".join(f"{d:>12}" for d in documents)
print(header)
print("-" * 90)

for phase in phases:
    row = f"{phase:<30}"
    for std_id in documents:
        count = coverage_matrix[std_id][phase]
        row += f"{count if count > 0 else '---':>12}"
    print(row)

In [ ]:
# Visualise the coverage matrix (repository encoding; not a table in either paper)
fig, ax = plt.subplots(figsize=(14, 5))

standards = list(coverage_matrix.keys())
phases = list(list(coverage_matrix.values())[0].keys())
data = np.array([[coverage_matrix[s][p] for p in phases] for s in standards])

im = ax.imshow(data, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(phases)))
ax.set_xticklabels(phases, rotation=30, ha='right')
ax.set_yticks(range(len(standards)))
ax.set_yticklabels(standards)

for i in range(len(standards)):
    for j in range(len(phases)):
        val = data[i, j]
        text = str(int(val)) if val > 0 else '---'
        color = 'white' if val > 3 else 'black'
        ax.text(j, i, text, ha='center', va='center', color=color, fontweight='bold')

plt.colorbar(im, label='Number of applicable clauses')
ax.set_title('Clause applicability per lifecycle phase (repository encoding)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3: GSN construction

The ISO/PAS 8800 Annex B pattern is the base structure: a top-level goal G1 and
five sub-goals G2 to G6. The integration attaches claims from the other
standards to existing goals where the assurance domain matches and creates new
goals only where no equivalent exists (WAISE paper, Step 3). The result has nine
goals:

- **G1** reformulated from Annex B G1 to cover the integrated safety and
  cybersecurity requirements; **G2 to G6** retained.
- **G7** added: SOTIF residual risk (ISO 21448).
- **G8** added: cybersecurity risk management (ISO/SAE 21434).
- **G9** added and undeveloped: modification re-assurance. It is undeveloped
  because no single standard prescribes the complete re-assurance workflow for a
  modified AI model; partial coverage exists.

A **junction point** is a goal where claims from two or more standards meet.
Step 4 examines these goals for decision points.

In [ ]:
# Step 3: Build the integrated GSN
gsn = build_integrated_gsn()
gsn_stats = gsn.compute_statistics()

print("Integrated GSN: element counts")
print("=" * 50)
for key, value in gsn_stats.items():
    print(f"  {key}: {value}")

print(f"\nJunction points (goals where claims from two or more documents meet,")
print(f"counted from each goal's source_standards):")
print("-" * 60)
for jp in gsn.get_junction_points():
    stds = ', '.join(jp.source_standards)
    print(f"  {jp.element_id}: [{stds}] ({len(jp.source_standards)} documents)")

print(f"\nUndeveloped goals:")
print("-" * 60)
for ug in gsn.get_undeveloped_goals():
    print(f"  {ug.element_id}: {ug.text}")

The repository records the standards behind each goal in two ways: the
`source_standards` list on each goal (used for the junction points above) and the
claim-to-goal mapping (used for the table below). The two can differ. At G8,
`source_standards` names ISO/SAE 21434 and ISO 26262, while only ISO/SAE 21434
claims map to G8. G9 is undeveloped and has no `source_standards`, while claims
from three documents map to it. At G3 the table also counts ISO/IEC TR 5469,
which is an informative technical report and not one of the four standards. `TRACEABILITY.md`,
[section 3](../TRACEABILITY.md#3-goals-of-the-integrated-pattern), records where the
two representations differ and why. The table lists goals only; claims that map
to the strategy S1 are left out.

In [ ]:
# Documents contributing claims per goal (repository encoding; not a table in either paper)
density = registry.compute_goal_density()

print("Documents contributing claims per GSN goal (repository encoding)")
print("=" * 75)

all_stds = list(list(density.values())[0].keys())
header = f"{'Goal':<6}" + "".join(f"{s:<14}" for s in all_stds) + "Active"
print(header)
print("-" * 75)

for goal, stds in density.items():
    if not goal.startswith('G'):
        continue  # strategies (S1) are not goals
    active = sum(1 for v in stds.values() if v)
    row = f"{goal:<6}"
    for s in all_stds:
        row += f"{'Y':<14}" if stds[s] else f"{'---':<14}"
    row += str(active)
    print(row)

print(f"\nG5, the V&V goal, is the goal where all four standards contribute claims.")
print(f"Decision point DP-2 (evidence-type asymmetry) sits there.")

### Claim-mapping and phase-coverage check

This check confirms that:
1. every extracted claim maps to at least one GSN node (or is recorded as outside
   the AI-component scope);
2. every lifecycle phase is covered by at least one goal;
3. no claim is left unmapped.

The WAISE paper describes this check (Section 6.3, Verification and Limitations).
It shows that the claims and phases are accounted for; it does not show that the
argument itself is complete.

In [ ]:
# Claim-mapping and phase-coverage check
from src.analysis.completeness import check_gsn_completeness

comp = check_gsn_completeness()

print("Claim-mapping and phase-coverage check")
print("=" * 70)
print(f"  Total claims across all documents:  {comp.total_claims}")
print(f"  Claims mapped to GSN nodes:        {comp.mapped_claims}")
print(f"  Unmapped claims:                   {len(comp.unmapped_claims)}")
print(f"  Lifecycle phases covered:          {len(comp.phases_covered)}/6")
print(f"  Goals without claims:              {', '.join(comp.goals_without_claims) or 'none'}")
print()
print("Claims per goal:")
for goal, count in sorted(comp.goals_with_claims.items()):
    marker = ' [UNDEVELOPED]' if count == 0 else ''
    print(f"  {goal}: {count} claims{marker}")
print()
status = 'PASSED' if comp.is_complete else 'FAILED'
print(f"Claim-mapping and phase-coverage check: {status}")
print(f"\nClaims recorded as outside the AI-component scope: "
      f"{comp.total_claims - comp.mapped_claims - len(comp.unmapped_claims)}")

---
## Step 4: Junction-point analysis

Each node where claims from two or more standards meet is examined for
**decision points**: places where the standards defer to application context
and the engineer must decide. The WAISE paper distinguishes three types:

- **Terminological (T):** different terms for the same concept
- **Methodological (M):** different methods for the same objective
- **Structural (S):** frameworks requiring decisions that no standard prescribes

The output is the decision-point catalogue (WAISE Table 4). The code writes
`I-1` to `I-7` for DP-1 to DP-7. It types DP-2 as structural, following the
paper's prose; Table 4 gives "S, M" (see `ERRATA.md`, section 1).

In [ ]:
# Step 4: decision-point catalogue (WAISE Table 4)
catalogue = DecisionPointCatalogue()

print("Decision points at junction points (WAISE Table 4)")
print("=" * 90)

inc_stats = catalogue.summary_statistics()
print(f"Total: {inc_stats['total']} ({inc_stats['structural']} structural, "
      f"{inc_stats['terminological']} terminological, {inc_stats['methodological']} methodological)")
print()

for inc in catalogue.inconsistencies:
    type_label = {'structural': 'S', 'terminological': 'T', 'methodological': 'M'}
    t = type_label[inc.inconsistency_type.value]
    nodes = ', '.join(inc.gsn_nodes)
    stds = ', '.join(inc.standards_involved)
    print(f"  {paper_id(inc.inconsistency_id)} ({inc.inconsistency_id} in the code) [{t}] at {nodes}")
    print(f"    Standards: {stds}")
    print(f"    {inc.description}")
    print()

In [ ]:
# Visualise decision points per GSN goal, by type
node_types = defaultdict(lambda: defaultdict(int))
for inc in catalogue.inconsistencies:
    for node in inc.gsn_nodes:
        node_types[node][inc.inconsistency_type.value] += 1

nodes = sorted(node_types.keys())
types = ['structural', 'terminological', 'methodological']
colors = ['#E74C3C', '#F39C12', '#3498DB']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(nodes))
width = 0.25

for i, (t, c) in enumerate(zip(types, colors)):
    values = [node_types[n][t] for n in nodes]
    ax.bar(x + i * width, values, width, label=t.capitalize(), color=c)

ax.set_xlabel('GSN goal')
ax.set_ylabel('Number of decision points')
ax.set_title('Decision points per GSN goal, by type', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(nodes)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 5: Integration-induced finding identification

The integrated GSN is examined for two categories of findings (WAISE paper,
Step 5):

- **Integration-induced findings** emerge only between the scopes of individual
  standards and are not detectable from any single standard in isolation.
- **Open methodological problems**: the standards prescribe a framework, but no
  published method specifies application-specific values for AI components.

The output is the findings classification (WAISE Table 5). The code writes
`Gap-1` to `Gap-5` for F-1 to F-5.

In [ ]:
# Step 5: findings classification (WAISE Table 5)
gap_cls = GapClassification()

print("Findings in the integrated argument (WAISE Table 5)")
print("=" * 90)

gap_stats = gap_cls.summary_statistics()
print(f"Total: {gap_stats['total']} findings")
print(f"  Integration-induced findings:  {gap_stats['integration_induced']}")
print(f"  Open methodological problems:  {gap_stats['total'] - gap_stats['integration_induced']}")
print()

for g in gap_cls.gaps:
    category = 'integration-induced finding' if g.integration_induced else 'open methodological problem'
    phases_listed = ', '.join(p.display_name for p in g.lifecycle_phases)
    print(f"  {paper_id(g.gap_id)} ({g.gap_id} in the code): {g.description}")
    print(f"    Category: {category} | Lifecycle phases: {phases_listed}")
    print()

In [ ]:
# Visualise findings by primary lifecycle phase (as the code records it)
phase_gaps = defaultdict(lambda: {'standard': 0, 'integration': 0})
for g in gap_cls.gaps:
    key = g.lifecycle_phase.display_name
    if g.integration_induced:
        phase_gaps[key]['integration'] += 1
    else:
        phase_gaps[key]['standard'] += 1

phases_with_gaps = sorted(phase_gaps.keys())
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(phases_with_gaps))

std_vals = [phase_gaps[p]['standard'] for p in phases_with_gaps]
int_vals = [phase_gaps[p]['integration'] for p in phases_with_gaps]

ax.bar(x - 0.15, std_vals, 0.3, label='Open methodological problem', color='#7B68EE')
ax.bar(x + 0.15, int_vals, 0.3, label='Integration-induced finding', color='#FF6347', hatch='//')

ax.set_xlabel('Primary lifecycle phase')
ax.set_ylabel('Number of findings')
ax.set_title('Findings by primary lifecycle phase', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(phases_with_gaps, rotation=15, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

### Severity scoring (repository analysis, not in either paper)

Neither paper scores findings. This repository scores each finding on four
dimensions (1 to 5) as an illustration for discussion:
- **Safety impact:** potential harm if the finding is not addressed
- **Likelihood of affecting a project:** how likely the finding leads to a real
  failure
- **Detectability:** difficulty of detecting the finding without integration
- **Remediation complexity:** effort needed to address the finding

In [ ]:
# Severity scoring (repository analysis, not in either paper)
severity = gap_cls.severity_scores()

print("Severity scoring of the findings (repository analysis, not in either paper)")
print("=" * 95)
print(f"{'Finding':<7} {'Safety':>7} {'Likely':>8} {'Detect':>7} {'Remed':>6} {'Overall':>8} {'Priority':>10} {'Int-Ind':>8}")
print("-" * 95)

for s in severity:
    ind = 'Yes' if s['integration_induced'] else 'No'
    print(f"{paper_id(s['gap_id']):<7} {s['safety_impact']:>7} {s['exploitability']:>8} "
          f"{s['detectability']:>7} {s['remediation_complexity']:>6} "
          f"{s['overall_severity']:>8.1f} {s['priority']:>10} {ind:>8}")

print()
critical = [s for s in severity if s['priority'] == 'CRITICAL']
print(f"Highest repository priority (overall score >= 4.0): {', '.join(paper_id(s['gap_id']) for s in critical)}")
print(f"Both are integration-induced findings.")

# Visualise severity scores
fig, ax = plt.subplots(figsize=(12, 5))
gap_ids = [paper_id(s['gap_id']) for s in severity]
dims = ['safety_impact', 'exploitability', 'detectability', 'remediation_complexity']
dim_labels = ['Safety impact', 'Likelihood of affecting a project', 'Detectability', 'Remediation']
x = np.arange(len(gap_ids))
width = 0.2
colors_sev = ['#E53935', '#FF9800', '#FFC107', '#42A5F5']

for i, (dim, label, color) in enumerate(zip(dims, dim_labels, colors_sev)):
    vals = [s[dim] for s in severity]
    ax.bar(x + i * width, vals, width, label=label, color=color, edgecolor='black', linewidth=0.5)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(gap_ids)
ax.set_ylabel('Score (1-5)')
ax.set_title('Severity scoring, repository analysis (higher = more severe)', fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylim(0, 6)
ax.axhline(y=4, color='red', linestyle='--', alpha=0.3)

for i, s in enumerate(severity):
    if s['integration_induced']:
        ax.annotate('INT', xy=(x[i] + 1.5 * width, 5.5), ha='center',
                    fontsize=8, fontweight='bold', color='#C62828')

plt.tight_layout()
plt.show()

### Counterfactual analysis (repository analysis, not in either paper)

The WAISE paper defines integration-induced findings as not detectable from any
single standard in isolation (Step 5), and its test suite checks that they do
not appear in any single standard's claim set. This section is the repository's
derivation supporting that definition (`src/analysis/counterfactual.py`).

For each standard it asks: if a project followed only this standard, which
findings would it see, and which would stay hidden? "Does not address" lists
what a standard is silent on. A standard is said to exclude something
explicitly only where its scope clause says so; of the four, that is
ISO 21448 Clause 1 for cybersecurity threats.

In [ ]:
# Counterfactual analysis (repository analysis, not in either paper)
from src.analysis.counterfactual import CounterfactualAnalysis

cf = CounterfactualAnalysis()

for p in cf.perspectives:
    print(f"{'=' * 80}")
    print(f"A project following only {p.standard_id} ({p.standard_name}):")
    print(f"{'=' * 80}")
    print(f"\n  Covers: {len(p.covers)} areas")
    for c in p.covers[:3]:
        print(f"    + {c}")
    if len(p.covers) > 3:
        print(f"    + ... and {len(p.covers) - 3} more")
    if p.explicitly_excludes:
        print(f"\n  Explicitly excludes (scope clause):")
        for e in p.explicitly_excludes:
            print(f"    - {e}")
    print(f"\n  Does not address:")
    for e in p.excludes:
        print(f"    - {e}")
    print(f"\n  Findings visible: {', '.join(paper_id(g.split(' (')[0]) for g in p.visible_gaps) or 'None'}")
    print(f"  Findings not visible: {', '.join(paper_id(g.split(' (')[0]) for g in p.invisible_gaps)}")
    print(f"\n  Not covered:")
    print(wrap(paper_id(p.blind_spots[0])))
    print()

In [ ]:
# Finding visibility matrix: which findings are visible from each standard
visibility = cf.get_gap_visibility_matrix()

print("Finding visibility matrix (repository analysis, not in either paper)")
print("=" * 75)
print(f"{'Standard':<18}", end='')
all_gaps = ['Gap-1', 'Gap-2', 'Gap-3', 'Gap-4', 'Gap-5']
for g in all_gaps:
    print(f"{paper_id(g):>8}", end='')
print()
print("-" * 75)

for std, gaps_vis in visibility.items():
    print(f"{std:<18}", end='')
    for g in all_gaps:
        mark = 'YES' if gaps_vis[g] else '---'
        print(f"{mark:>8}", end='')
    print()

print()
print("F-3 and F-4 are not visible from any single standard;")
print("they are visible only in the row labelled Integrated.")

# Visualise as heatmap
fig, ax = plt.subplots(figsize=(10, 4))
stds_list = list(visibility.keys())
data_vis = np.array([[1 if visibility[s][g] else 0 for g in all_gaps] for s in stds_list])

im = ax.imshow(data_vis, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(all_gaps)))
ax.set_xticklabels([paper_id(g) for g in all_gaps])
ax.set_yticks(range(len(stds_list)))
ax.set_yticklabels(stds_list)

for i in range(len(stds_list)):
    for j in range(len(all_gaps)):
        text = 'Visible' if data_vis[i, j] else 'Hidden'
        color = 'black' if data_vis[i, j] else 'white'
        ax.text(j, i, text, ha='center', va='center', fontsize=8, color=color, fontweight='bold')

ax.set_title('Which findings are visible from each standard? (repository analysis)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Evidence-type asymmetry at G5 (decision point DP-2)

G5, the verification and validation (V&V) goal, is the goal where all four
standards contribute claims. The case-study evidence set at G5 includes one
evidence type per contributing standard:

- **(a)** MC/DC structural coverage of the surrounding non-AI software (ISO 26262)
- **(b)** SOTIF triggering-condition coverage in scenario testing (ISO 21448)
- **(c)** deep-ensemble disagreement as uncertainty indicator, measured by AUROC
  (ISO/PAS 8800)
- **(d)** adversarial penetration testing against LiDAR spoofing and
  perturbation, measured by attack success rate (ISO/SAE 21434)

The evidence types are disjoint by design, each on its own measurement scale,
and no standard prescribes how they combine into a single sufficiency claim at
G5. The WAISE abstract reports this evidence asymmetry. The printout below lists
the scales as the code encodes them.

In [ ]:
# Evidence types at G5 (decision point DP-2)
convergence = EvidenceConvergenceAnalysis()

print("Evidence types (a) to (d) at G5, the V&V goal")
print("=" * 80)

for i, et in zip('abcd', convergence.evidence_types):
    print(f"\n  ({i}) {et.name}")
    print(f"      Standard:    {et.standard} {et.clause}")
    print(f"      Measures:    {et.what_measured}")
    print(f"      Scale:       {et.scale}")
    print(f"      Instance:")
    print(wrap(et.case_study_instance, indent=8))

print(f"\n{'=' * 80}")
print(f"DP-2: the evidence types are disjoint by design, each on its own")
print(f"measurement scale, and no standard prescribes how they combine")
print(f"into a single sufficiency claim at G5.")

In [ ]:
# Evidence types at G5 (repository rendering; the WAISE paper's equivalent is Fig. 1(b))
from src.visualization.coverage_plots import plot_evidence_convergence

plot_evidence_convergence(None)
plt.gca().set_title("Evidence types at G5 (repository rendering; the WAISE paper's equivalent is Fig. 1(b))",
                    fontsize=12, fontweight='bold', pad=20)
plt.show()

### Evidence-combination sketch (repository analysis, not in either paper)

The WAISE paper lists the combination method as future work. This repository
sketches three possible approaches for discussion: a threshold per evidence
type, Dempster-Shafer (D-S) evidence theory, and multi-criteria decision
analysis (MCDA). None of them is a result of either paper.

In [ ]:
# Evidence-combination sketch (repository analysis, not in either paper)
sketch = convergence.get_solution_sketch()

print("Repository sketches for evidence combination at G5 (not in either paper)")
print("=" * 80)
print()

for i, approach in enumerate(sketch['approaches'], 1):
    print(f"{'_' * 80}")
    print(f"  Approach {i}: {approach['name']}")
    print(f"  Feasibility: {approach['feasibility']}")
    print(f"{'_' * 80}")
    print(wrap(approach['description'], indent=2))
    print(f"  Pros:")
    for p in approach['pros']:
        print(f"    + {p}")
    print(f"  Cons:")
    for c in approach['cons']:
        print(f"    - {c}")
    print()

print(f"{'=' * 80}")
print("The WAISE paper leaves the combination method to future work;")
print("these approaches are repository sketches for discussion.")

---
## Seeded illustration of the weather-grid evaluation

A seeded synthetic illustration of the evaluation pipeline over a 5x5 rain and
visibility grid (25 weather conditions). No detector is run and CARLA is not
connected; the numbers are not measurements (see
`data/synthetic_illustrations/README.md`). The WAISE paper's CARLA-based
evidence comes from its two companion studies (WAISE paper, Section 3.1), not
from this code.

**Case-study thresholds for potential triggering conditions:** rain > 20 mm/h
**or** visibility < 200 m. These are a project choice for this case study;
ISO 21448 sets no numeric weather values.

The illustration shows the format of the output: recall and ensemble divergence
per weather condition, split by whether the condition meets the case-study
thresholds. The lower recall under those conditions is built into the generating
formula.

In [ ]:
# Run the seeded illustration (no detector, no CARLA; not a measurement)
eval_result = generate_synthetic_illustration(
    num_scenes_per_weather=SCENES_PER_WEATHER,
    seed=SEED,
)
summary = eval_result.compute_summary()

print(f"Synthetic weather-grid illustration (seed {SEED}; not a measurement)")
print("=" * 60)
print(f"  Weather conditions:            {summary['total_weather_conditions']}")
print(f"  Meeting case-study thresholds: {summary['triggering_conditions']}")
print(f"  Not meeting thresholds:        {summary['total_weather_conditions'] - summary['triggering_conditions']}")
print()
print(f"  Overall mean recall:           {summary['overall_mean_recall']:.4f}")
print(f"  Recall, meets thresholds:      {summary['triggering_mean_recall']:.4f}")
print(f"  Recall, below thresholds:      {summary['non_triggering_mean_recall']:.4f}")
print(f"  Illustrated recall gap:        {summary['non_triggering_mean_recall'] - summary['triggering_mean_recall']:.4f}")
print()
print(f"  Overall divergence:            {summary['overall_mean_divergence']:.4f}")
print(f"  Divergence, meets thresholds:  {summary['triggering_mean_divergence']:.4f}")
print(f"  Total false negatives:         {summary['total_false_negatives']}")

In [ ]:
# Illustrated recall and divergence across the 5x5 grid (synthetic)
from src.visualization.coverage_plots import plot_weather_heatmap

plot_weather_heatmap(eval_result.weather_results, None)
plt.gcf().suptitle(f'Synthetic weather-grid illustration, seed {SEED} (not a measurement)\n'
                   '(red border = potential triggering condition under the case-study thresholds)',
                   fontsize=11, fontweight='bold')
plt.show()

In [ ]:
# Conditions meeting vs not meeting the case-study thresholds (synthetic)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

categories = ['Below thresholds', 'Meets thresholds']
recalls = [summary['non_triggering_mean_recall'], summary['triggering_mean_recall']]

ax1.bar(categories, recalls, color=['#2ECC71', '#E67E22'], width=0.5, edgecolor='black')
ax1.set_ylabel('Mean Recall')
ax1.set_title('Illustrated recall by weather category', fontweight='bold')
ax1.set_ylim(0, 1)
for i, v in enumerate(recalls):
    ax1.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Compute actual non-triggering divergence
non_trig_divs = [wr.mean_divergence for wr in eval_result.weather_results if not wr.weather.sotif_triggering]
trig_divs = [wr.mean_divergence for wr in eval_result.weather_results if wr.weather.sotif_triggering]
divs = [np.mean(non_trig_divs), np.mean(trig_divs)]
ax2.bar(categories, divs, color=['#2ECC71', '#E67E22'], width=0.5, edgecolor='black')
ax2.set_ylabel('Mean Geometric Divergence')
ax2.set_title('Illustrated ensemble divergence by weather category', fontweight='bold')
ax2.set_ylim(0, 1)
for i, v in enumerate(divs):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle(f'Synthetic weather-grid illustration (seed {SEED}): case-study thresholds met or not', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed per-weather results table
print(f"Per-weather-condition output of the synthetic illustration (seed {SEED})")
print("=" * 100)
print(f"{'Weather':<28} {'Rain':>6} {'Fog':>5} {'Meets':>5} {'Recall':>8} {'Prec':>8} {'Div':>8} {'FN':>5}")
print("-" * 100)

for wr in eval_result.weather_results:
    trig = 'Yes' if wr.weather.sotif_triggering else 'No'
    print(f"{wr.weather.name:<28} {wr.weather.rain_intensity:>5.0f} "
          f"{wr.weather.fog_density:>5.0f} {trig:>5} "
          f"{wr.mean_recall:>8.4f} {wr.mean_precision:>8.4f} "
          f"{wr.mean_divergence:>8.4f} {wr.total_false_negatives:>5}")

### Seed check of the illustration (repository analysis, not in either paper)

The illustration is run with five noise seeds to show that the illustrated
degradation, which is built into the generating formula, is not an artefact of
the noise seed. The module's printed "Conclusion" line refers to this
illustration only.

In [ ]:
# Seed check of the illustration (repository analysis, not in either paper)
from src.analysis.sensitivity import run_sensitivity_analysis, print_sensitivity_report

sens_result = run_sensitivity_analysis(
    seeds=[42, 123, 256, 512, 1024],
    scenes_per_weather=SCENES_PER_WEATHER,
)
print_sensitivity_report(sens_result)

# Visualise the illustrated recall gap across noise seeds
fig, ax = plt.subplots(figsize=(10, 4))
seeds = sens_result.seeds
x = np.arange(len(seeds))

ax.bar(x - 0.15, sens_result.non_triggering_recall, 0.3,
       label='Below thresholds', color='#2ECC71', edgecolor='black')
ax.bar(x + 0.15, sens_result.triggering_recall, 0.3,
       label='Meets thresholds', color='#E67E22', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in seeds])
ax.set_xlabel('Noise seed')
ax.set_ylabel('Illustrated mean recall')
ax.set_title('Illustrated recall gap across noise seeds (synthetic)', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)

for i, gap in enumerate(sens_result.recall_gap):
    mid = (sens_result.non_triggering_recall[i] + sens_result.triggering_recall[i]) / 2
    ax.annotate(f'{gap:.3f}', xy=(i, mid), ha='center', fontsize=8, fontweight='bold', color='#C62828')

plt.tight_layout()
plt.show()

### Threshold sensitivity of the illustration (repository analysis, not in either paper)

The case-study default (rain > 20 mm/h or visibility < 200 m) is a project
choice; ISO 21448 sets no numeric weather values. This cell varies both
thresholds and reports the illustrated recall gap at each combination.

Some rows and columns repeat. The grid has discrete levels (rain 0, 10, 25,
50 and 100 mm/h; visibility 10, 50, 100, 200 and 500 m), so thresholds that
fall between the same two levels select the same conditions and give the same
recall gap. With the rule rain > threshold, 10, 15 and 20 mm/h all select 25 mm/h
and above, and 25, 30 and 40 mm/h all select 50 mm/h and above. With the rule
visibility < threshold, 150 and 200 m both select 100 m and below, and 250 and
300 m both select 200 m and below.

In [ ]:
# Threshold sensitivity of the illustration (repository analysis, not in either paper)
from src.analysis.sensitivity import run_threshold_sensitivity

thresh_result = run_threshold_sensitivity(seed=SEED, scenes_per_weather=SCENES_PER_WEATHER)

print("Threshold sensitivity of the synthetic illustration")
print("=" * 80)
print(f"Case-study default (a project choice; ISO 21448 sets no numeric weather values): rain > 20 mm/h OR visibility < 200 m")
print()

header = f"{'Rain>':<10}"
for vis in thresh_result.visibility_thresholds:
    header += f"  vis<{vis:>3}m"
print(header)
print("-" * len(header))

for i, rain in enumerate(thresh_result.rain_thresholds):
    row = f"{rain:>5}mm/h "
    for j in range(len(thresh_result.visibility_thresholds)):
        gap = thresh_result.recall_gaps[i][j]
        marker = " *" if (rain == 20 and thresh_result.visibility_thresholds[j] == 200) else "  "
        row += f"  {gap:>6.3f}{marker}"
    print(row)

print()
print("  * = case-study default")
print("  Positive values = lower illustrated recall for conditions meeting the thresholds")

# Visualise as heatmap
fig, ax = plt.subplots(figsize=(10, 6))
gap_data = np.array(thresh_result.recall_gaps)
im = ax.imshow(gap_data, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(thresh_result.visibility_thresholds)))
ax.set_xticklabels([f'<{v}m' for v in thresh_result.visibility_thresholds])
ax.set_yticks(range(len(thresh_result.rain_thresholds)))
ax.set_yticklabels([f'>{r}mm/h' for r in thresh_result.rain_thresholds])
ax.set_xlabel('Visibility threshold')
ax.set_ylabel('Rain threshold')

for i in range(len(thresh_result.rain_thresholds)):
    for j in range(len(thresh_result.visibility_thresholds)):
        val = gap_data[i, j]
        marker = '\n(case-study default)' if (thresh_result.rain_thresholds[i] == 20 and thresh_result.visibility_thresholds[j] == 200) else ''
        ax.text(j, i, f'{val:.3f}{marker}', ha='center', va='center', fontsize=8)

plt.colorbar(im, label='Illustrated recall gap (below thresholds minus meets thresholds)')
ax.set_title('Illustrated recall gap against the case-study thresholds (synthetic)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Traceability matrix

A consolidated view linking every GSN goal to its contributing standards,
decision points and findings.

In [ ]:
# Traceability matrix
from src.analysis.traceability import TraceabilityMatrix

trace = TraceabilityMatrix()
trace_summary = trace.get_summary()

print("Traceability matrix: GSN goals -> claims, decision points, findings")
print("=" * 100)
def goals(n):
    return f"{n} goal" if n == 1 else f"{n} goals"

print(f"Overall: {goals(trace_summary['covered'])} with no decision point or finding [+], "
      f"{trace_summary['partial']} with a decision point only [~], {trace_summary['gap']} with a finding or no claims [!] "
      f"(of {goals(trace_summary['total_goals'])})")
print()

for e in trace.entries:
    status_marker = {"covered": "+", "partial": "~", "gap": "!"}[e.status]
    print(f"  [{status_marker}] {e.goal_id}: "
          f"{textwrap.shorten(' '.join(e.goal_text.split()), width=80, placeholder=' ...')}")
    print(f"      Standards: {', '.join(e.contributing_standards) or 'NONE'} "
          f"({e.num_claims} claims)")
    if e.inconsistencies:
        print(f"      Decision points: {paper_id(', '.join(e.inconsistencies))}")
    if e.gaps:
        gap_str = paper_id(", ".join(e.gaps))
        if e.integration_induced_gaps:
            gap_str += f" (integration-induced: {paper_id(', '.join(e.integration_induced_gaps))})"
        print(f"      Findings: {gap_str}")
    print()

---
## Comparison with related work (repository analysis, not in either paper)

This repository compares the method with three approaches it selected: AMLAS,
SACE and STPA-SafeSec (Friedberg et al., "STPA-SafeSec: Safety and security
analysis for cyber-physical systems", Journal of Information Security and
Applications 34, 2017, with a power-grid case study). The WAISE paper does not
cite STPA-SafeSec. The paper's own comparison is its Table 1, which covers ten
works; the claim it makes there is that no published work integrates the
applicable standards into a single GSN argument with cybersecurity claims.

The limitation given for this work is the WAISE paper's own (Section 6.3).

In [ ]:
# Related-work comparison (repository analysis, not in either paper)
from src.analysis.completeness import get_related_work_comparison

approaches = get_related_work_comparison()

print("Comparison with three related approaches (repository analysis, not in either paper)")
print("=" * 100)

for a in approaches:
    print(f"\n  {a.name} ({a.authors}, {a.year})")
    print(f"    Documents:        {', '.join(a.standards_covered)}")
    print(f"    Uses GSN:         {'Yes' if a.uses_gsn else 'No'}")
    print(f"    Identifies integration-induced findings: "
          f"{'Yes' if a.identifies_integration_gaps else 'No'}")
    print(f"    Case-study domain: {a.case_study_domain}")
    print(f"    Main limitation:")
    print(wrap(paper_id(a.key_limitation), indent=6))

print(f"\n{'=' * 100}")
print("The WAISE paper's claim (Section 2, Table 1): no published work integrates")
print("the applicable standards into a single GSN argument with cybersecurity claims.")

---
## Summary of the outputs

The printout collects the outputs of the five steps and labels the repository
analyses as such.

In [ ]:
# Summary of the outputs
print("SUMMARY OF THE OUTPUTS")
print("=" * 70)
print()
print(f"1. CLAIM EXTRACTION (STEP 1)")
print(f"   {total_claims} claims extracted from {total_clauses} clauses")
print(f"   across {len(registry.all_standards)} documents (four standards and one informative technical report)")
print()
print(f"2. INTEGRATED GSN (STEP 3)")
print(f"   {gsn_stats['goals']} goals (G1 reformulated, G2 to G6 retained from Annex B, G7 to G9 added)")
print(f"   {gsn_stats['junction_points']} junction points")
print(f"   {gsn_stats['solutions']} solution nodes")
print(f"   Claim-mapping and phase-coverage check: {'PASSED' if comp.is_complete else 'FAILED'}")
print()
print(f"3. DECISION POINTS (STEP 4, WAISE TABLE 4)")
print(f"   {inc_stats['total']} total: {inc_stats['structural']} structural, "
      f"{inc_stats['terminological']} terminological, {inc_stats['methodological']} methodological")
print()
print(f"4. FINDINGS (STEP 5, WAISE TABLE 5)")
print(f"   {gap_stats['total']} findings: {gap_stats['integration_induced']} integration-induced "
      f"({', '.join(paper_id(g.gap_id) for g in gap_cls.gaps if g.integration_induced)}), "
      f"{gap_stats['total'] - gap_stats['integration_induced']} open methodological problems "
      f"({', '.join(paper_id(g.gap_id) for g in gap_cls.gaps if not g.integration_induced)})")
print(f"   Repository severity scoring (not in either paper), highest priority: "
      f"{', '.join(paper_id(s['gap_id']) for s in severity if s['priority'] == 'CRITICAL')}")
print()
print(f"5. EVIDENCE-TYPE ASYMMETRY AT G5 (DECISION POINT DP-2)")
print(f"   Four evidence types, disjoint by design, each on its own measurement scale")
print(f"   No standard prescribes how they combine into a single sufficiency claim")
print(f"   Repository sketch (not in either paper): threshold per type, Dempster-Shafer, MCDA")
print()
print(f"6. SYNTHETIC ILLUSTRATION (NOT A MEASUREMENT; NO DETECTOR, NO CARLA)")
print(f"   Illustrated recall, meets thresholds: {summary['triggering_mean_recall']:.4f} "
      f"vs below thresholds: {summary['non_triggering_mean_recall']:.4f}")
print(f"   Illustrated recall gap: {summary['non_triggering_mean_recall'] - summary['triggering_mean_recall']:.1%}")
print(f"   Illustrated divergence, meets thresholds: {summary['triggering_mean_divergence']:.4f}")
print(f"   Seed check across {len(sens_result.seeds)} noise seeds (gap std: {sens_result.std_recall_gap:.4f})")
print()
print(f"7. COUNTERFACTUAL ANALYSIS (REPOSITORY ANALYSIS, NOT IN EITHER PAPER)")
print(f"   F-3 and F-4 are not visible from any of the four standards alone")
print(f"   They appear only in the integrated argument")

---
## Write the repository's output files

Run this cell to write the output files (JSON, CSV, LaTeX tables, figures) to
the `output/` directory. Neither paper includes these files. The reproduction
entry point is `make reproduce` (see `REPRODUCING.md`).

In [ ]:
# Write the repository's output files
from src.results.latex_tables import generate_all_tables
from src.results.export import export_json, export_csv_tables, export_summary_report

os.makedirs('output/csv', exist_ok=True)
os.makedirs('output/latex', exist_ok=True)
os.makedirs('output/figures', exist_ok=True)

# JSON
json_path = export_json(registry, eval_result, 'output')
print(f"JSON:   {json_path}")

# CSV
csv_files = export_csv_tables(registry, eval_result, 'output/csv')
for cf in csv_files:
    print(f"CSV:    {cf}")

# LaTeX
tables = generate_all_tables(summary, 'output/latex')
for name in tables:
    print(f"LaTeX:  output/latex/{name}.tex")

# Report
report_path = export_summary_report(registry, eval_result, 'output')
print(f"Report: {report_path}")

print("\nRepository output files written.")